# FR3 `fr3_link0` 기준 Sampled Reachability Map

이 노트북은 `sampled_reachability_maps`의 FK 샘플링 방식을 FR3에 맞게 구성한 것입니다.

생성 결과:
- 기준 좌표계: `fr3_link0`
- 엔드이펙터: `fr3_hand_tcp` (없으면 `fr3_link8`)
- 6D pose: `[x, y, z, roll, pitch, yaw]`
- voxel별 방문 횟수
- voxel별 최대 Yoshikawa manipulability
- 시각화용 3D 요약 map

> 주의: 현재 버전은 자기충돌, 이동 데크, 바닥 및 환경 충돌을 아직 필터링하지 않습니다.

## 0. Colab GPU 설정

상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**을 선택한 후 아래 셀부터 순서대로 실행합니다.

In [ ]:
# 필요한 패키지 설치
%pip -q install pytorch-kinematics robot-descriptions yourdfpy xacrodoc

print("패키지 설치 완료")

## 1. 실행 장치와 실험 설정

처음에는 10만 샘플과 45도 방향 해상도로 전체 과정이 정상 동작하는지 확인합니다. 이후 `NUM_SAMPLES`를 늘리고 `ANGULAR_RES_DEG`를 22.5도로 바꿉니다.

In [ ]:
import math
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib.pyplot as plt
import torch
import pytorch_kinematics as pk

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
SEED = 42

# 검증용 설정
NUM_SAMPLES = 100_000
BATCH_SIZE = 5_000
CARTESIAN_RES_M = 0.05
ANGULAR_RES_DEG = 45.0
ANGULAR_RES_RAD = math.radians(ANGULAR_RES_DEG)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Samples:", f"{NUM_SAMPLES:,}")
print("Position resolution:", CARTESIAN_RES_M, "m")
print("Angular resolution:", ANGULAR_RES_DEG, "deg")

## 2. 공식 FR3 URDF 생성

`robot_descriptions`가 Franka 공식 description 저장소를 내려받고 Xacro를 URDF로 변환합니다.

In [ ]:
from robot_descriptions.loaders.yourdfpy import load_robot_description

fr3_robot = load_robot_description("fr3_description")
urdf_path = Path("/content/fr3_generated.urdf")
fr3_robot.write_xml_file(str(urdf_path))

# XML encoding 선언 문제를 피하기 위해 bytes로 읽습니다.
urdf_bytes = urdf_path.read_bytes()
urdf_root = ET.fromstring(urdf_bytes)
link_names = [link.attrib["name"] for link in urdf_root.findall("link")]

if "fr3_hand_tcp" in link_names:
    END_EFFECTOR_LINK = "fr3_hand_tcp"
elif "fr3_link8" in link_names:
    END_EFFECTOR_LINK = "fr3_link8"
else:
    raise RuntimeError("fr3_hand_tcp 또는 fr3_link8을 URDF에서 찾지 못했습니다.")

ROOT_LINK = "fr3_link0"

print("URDF:", urdf_path)
print("Links:", len(link_names))
print("Root link:", ROOT_LINK)
print("End-effector link:", END_EFFECTOR_LINK)

## 3. `fr3_link0 → TCP` 기구학 체인과 관절 한계

관절 한계는 별도로 하드코딩하지 않고 공식 URDF에서 읽습니다.

In [ ]:
chain = pk.build_serial_chain_from_urdf(
    urdf_bytes,
    END_EFFECTOR_LINK,
    root_link_name=ROOT_LINK,
).to(device=DEVICE, dtype=DTYPE)

joint_names = chain.get_joint_parameter_names()
if len(joint_names) != 7:
    raise RuntimeError(f"FR3 arm joint가 7개가 아닙니다: {joint_names}")

joint_xml = {
    joint.attrib["name"]: joint
    for joint in urdf_root.findall("joint")
}

q_min_values = []
q_max_values = []
for name in joint_names:
    limit = joint_xml[name].find("limit")
    if limit is None or "lower" not in limit.attrib or "upper" not in limit.attrib:
        raise RuntimeError(f"{name}의 관절 한계를 찾지 못했습니다.")
    q_min_values.append(float(limit.attrib["lower"]))
    q_max_values.append(float(limit.attrib["upper"]))

q_min = torch.tensor(q_min_values, dtype=DTYPE, device=DEVICE)
q_max = torch.tensor(q_max_values, dtype=DTYPE, device=DEVICE)

print("Joint limits from official URDF")
for i, name in enumerate(joint_names):
    print(
        f"{name}: {q_min[i].item(): .4f} ~ {q_max[i].item(): .4f} rad "
        f"({math.degrees(q_min[i].item()): .1f} ~ {math.degrees(q_max[i].item()): .1f} deg)"
    )

## 4. 관절 샘플링, FK, Jacobian, Manipulability

각 batch에서 관절각을 균등 샘플링하고 TCP pose와 Jacobian을 계산합니다. Yoshikawa manipulability는 `sqrt(det(J J^T))`로 계산합니다.

In [ ]:
pose_chunks = []
manipulability_chunks = []

start_time = time.time()

with torch.no_grad():
    for start in range(0, NUM_SAMPLES, BATCH_SIZE):
        end = min(start + BATCH_SIZE, NUM_SAMPLES)
        current_batch_size = end - start

        random_unit = torch.rand(
            (current_batch_size, 7),
            dtype=DTYPE,
            device=DEVICE,
        )
        q_batch = q_min + random_unit * (q_max - q_min)

        transforms = chain.forward_kinematics(q_batch).get_matrix()
        positions = transforms[:, :3, 3]
        rotations = transforms[:, :3, :3]
        euler_xyz = pk.transforms.matrix_to_euler_angles(rotations, "XYZ")
        poses_6d = torch.cat([positions, euler_xyz], dim=1)

        jacobian = chain.jacobian(q_batch)
        jj_t = jacobian @ jacobian.transpose(-1, -2)
        determinant = torch.linalg.det(jj_t)
        manipulability = torch.sqrt(torch.clamp(determinant, min=0.0))

        pose_chunks.append(poses_6d.cpu())
        manipulability_chunks.append(manipulability.cpu())

        print(f"진행: {end:>8,} / {NUM_SAMPLES:,}")

poses_6d_all = torch.cat(pose_chunks, dim=0)
manipulability_all = torch.cat(manipulability_chunks, dim=0)

if not torch.isfinite(poses_6d_all).all():
    raise RuntimeError("6D pose에 NaN 또는 Inf가 있습니다.")
if not torch.isfinite(manipulability_all).all():
    raise RuntimeError("Manipulability에 NaN 또는 Inf가 있습니다.")

print("\n계산 완료")
print("6D poses:", poses_6d_all.shape)
print("Manipulability:", manipulability_all.shape)
print(f"Elapsed: {time.time() - start_time:.2f} s")

## 5. Sparse 6D voxel map 생성

원본 저장소처럼 위치와 방향을 voxel로 이산화하되, 전체 dense 배열을 만들지 않고 실제로 방문한 voxel만 저장합니다.

In [ ]:
# 위치 voxel index: 원점은 fr3_link0
position_indices = torch.floor(
    poses_6d_all[:, :3] / CARTESIAN_RES_M
).to(torch.int64)

# Euler angle 정규화
angles = poses_6d_all[:, 3:].clone()
angles[:, 0] = torch.remainder(angles[:, 0] + math.pi, 2 * math.pi) - math.pi
angles[:, 2] = torch.remainder(angles[:, 2] + math.pi, 2 * math.pi) - math.pi
angles[:, 1] = torch.clamp(
    angles[:, 1],
    min=-math.pi / 2,
    max=math.pi / 2 - 1e-6,
)

angle_min = torch.tensor(
    [-math.pi, -math.pi / 2, -math.pi],
    dtype=torch.float32,
)
angle_bin_counts = torch.tensor(
    [
        math.ceil(2 * math.pi / ANGULAR_RES_RAD),
        math.ceil(math.pi / ANGULAR_RES_RAD),
        math.ceil(2 * math.pi / ANGULAR_RES_RAD),
    ],
    dtype=torch.int64,
)

orientation_indices = torch.floor(
    (angles - angle_min) / ANGULAR_RES_RAD
).to(torch.int64)
orientation_indices = torch.maximum(
    orientation_indices,
    torch.zeros_like(orientation_indices),
)
orientation_indices = torch.minimum(
    orientation_indices,
    angle_bin_counts - 1,
)

sample_voxel_indices_6d = torch.cat(
    [position_indices, orientation_indices],
    dim=1,
)

(
    voxel_indices_6d,
    sample_to_voxel,
    visitation_counts_6d,
) = torch.unique(
    sample_voxel_indices_6d,
    dim=0,
    return_inverse=True,
    return_counts=True,
)

max_manipulability_6d = torch.zeros(
    voxel_indices_6d.shape[0],
    dtype=torch.float32,
)
max_manipulability_6d.scatter_reduce_(
    dim=0,
    index=sample_to_voxel,
    src=manipulability_all.float(),
    reduce="amax",
    include_self=True,
)

position_centers_6d = (
    voxel_indices_6d[:, :3].float() + 0.5
) * CARTESIAN_RES_M
orientation_centers_6d = angle_min + (
    voxel_indices_6d[:, 3:].float() + 0.5
) * ANGULAR_RES_RAD
voxel_poses_6d = torch.cat(
    [position_centers_6d, orientation_centers_6d],
    dim=1,
)

print("Joint samples:", f"{NUM_SAMPLES:,}")
print("Visited 6D voxels:", f"{voxel_indices_6d.shape[0]:,}")
print("Max visitation count:", visitation_counts_6d.max().item())
print("Max manipulability:", max_manipulability_6d.max().item())

## 6. 시각화용 3D 요약 map과 파일 저장

방향을 합쳐 같은 위치 voxel의 방문 횟수 합과 최대 manipulability를 저장합니다.

In [ ]:
(
    voxel_indices_3d,
    sample_to_voxel_3d,
    visitation_counts_3d,
) = torch.unique(
    position_indices,
    dim=0,
    return_inverse=True,
    return_counts=True,
)

max_manipulability_3d = torch.zeros(
    voxel_indices_3d.shape[0],
    dtype=torch.float32,
)
max_manipulability_3d.scatter_reduce_(
    dim=0,
    index=sample_to_voxel_3d,
    src=manipulability_all.float(),
    reduce="amax",
    include_self=True,
)

voxel_centers_3d = (
    voxel_indices_3d.float() + 0.5
) * CARTESIAN_RES_M

map_6d_path = Path("/content/fr3_link0_reachability_6d.pt")
map_3d_path = Path("/content/fr3_link0_reachability_3d.pt")

common_metadata = {
    "robot": "fr3",
    "frame_id": ROOT_LINK,
    "end_effector_link": END_EFFECTOR_LINK,
    "joint_names": joint_names,
    "joint_position_min": q_min.cpu(),
    "joint_position_max": q_max.cpu(),
    "num_joint_samples": NUM_SAMPLES,
    "seed": SEED,
    "cartesian_resolution_m": CARTESIAN_RES_M,
    "collision_filtered": False,
}

torch.save(
    {
        **common_metadata,
        "angular_resolution_rad": ANGULAR_RES_RAD,
        "angular_resolution_deg": ANGULAR_RES_DEG,
        "euler_convention": "XYZ",
        "voxel_indices_6d": voxel_indices_6d,
        "voxel_poses_6d": voxel_poses_6d,
        "visitation_counts": visitation_counts_6d,
        "max_manipulability": max_manipulability_6d,
    },
    map_6d_path,
)

torch.save(
    {
        **common_metadata,
        "voxel_indices_3d": voxel_indices_3d,
        "voxel_centers_3d": voxel_centers_3d,
        "visitation_counts": visitation_counts_3d,
        "max_manipulability": max_manipulability_3d,
    },
    map_3d_path,
)

print("3D voxels:", f"{voxel_indices_3d.shape[0]:,}")
print("Saved:", map_6d_path)
print("Saved:", map_3d_path)

In [ ]:
# 3D 위치 map 시각화: 색상은 log(1 + 방문 횟수)
max_plot_points = 20_000
if voxel_centers_3d.shape[0] > max_plot_points:
    plot_indices = torch.argsort(
        visitation_counts_3d,
        descending=True,
    )[:max_plot_points]
else:
    plot_indices = torch.arange(voxel_centers_3d.shape[0])

plot_points = voxel_centers_3d[plot_indices].numpy()
plot_scores = np.log1p(visitation_counts_3d[plot_indices].numpy())

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    plot_points[:, 0],
    plot_points[:, 1],
    plot_points[:, 2],
    c=plot_scores,
    cmap="plasma",
    s=7,
    alpha=0.55,
)
ax.scatter(0, 0, 0, color="red", s=100, label="fr3_link0")
ax.set_title(
    f"FR3 Reachability Map in fr3_link0\n"
    f"{CARTESIAN_RES_M * 100:.0f} cm, {NUM_SAMPLES:,} samples"
)
ax.set_xlabel("X [m]")
ax.set_ylabel("Y [m]")
ax.set_zlabel("Z [m]")
ax.legend()
fig.colorbar(scatter, ax=ax, shrink=0.7, label="log(1 + visitation count)")
plt.tight_layout()
plt.show()

## 7. 평면 투영으로 보기

3D 맵을 위에서 본 **XY**, 옆에서 본 **XZ**, 정면에서 본 **YZ** 평면으로 투영합니다. 여러 높이 또는 깊이의 voxel이 같은 평면 격자에 겹치면 방문 횟수를 합산합니다.

In [ ]:
# 3D voxel들을 지정한 2개 축으로 투영하고 방문 횟수를 합산하는 함수
def make_2d_projection(axis_indices):
    indices_2d = voxel_indices_3d[:, axis_indices]
    unique_indices_2d, inverse = torch.unique(
        indices_2d,
        dim=0,
        return_inverse=True,
    )

    projected_counts = torch.zeros(
        unique_indices_2d.shape[0],
        dtype=torch.int64,
    )
    projected_counts.scatter_add_(
        0,
        inverse,
        visitation_counts_3d.to(torch.int64),
    )

    projected_centers = (
        unique_indices_2d.float() + 0.5
    ) * CARTESIAN_RES_M
    return projected_centers.numpy(), projected_counts.numpy()


projections = [
    ((0, 1), "Top view: XY", "X [m]", "Y [m]"),
    ((0, 2), "Side view: XZ", "X [m]", "Z [m]"),
    ((1, 2), "Front view: YZ", "Y [m]", "Z [m]"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
last_scatter = None

for ax, (axis_indices, title, xlabel, ylabel) in zip(axes, projections):
    points_2d, counts_2d = make_2d_projection(axis_indices)
    last_scatter = ax.scatter(
        points_2d[:, 0],
        points_2d[:, 1],
        c=np.log1p(counts_2d),
        cmap="plasma",
        s=16,
        marker="s",
    )
    ax.scatter(0, 0, color="red", s=70, label="fr3_link0")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    f"FR3 Reachability Map — 2D Projections\n"
    f"{CARTESIAN_RES_M * 100:.0f} cm, {NUM_SAMPLES:,} samples"
)
fig.colorbar(
    last_scatter,
    ax=axes,
    shrink=0.8,
    label="log(1 + projected visitation count)",
)
plt.show()

## 8. 결과 다운로드

아래 셀을 실행하면 6D map과 시각화용 3D map을 현재 컴퓨터로 내려받습니다.

In [ ]:
from google.colab import files

files.download(str(map_6d_path))
files.download(str(map_3d_path))

## 다음 단계

1. 이동 데크 위 `fr3_link0`의 실제 바닥 기준 높이를 확인합니다.
2. 페트병 파지점 높이를 `fr3_link0` 기준 Z 범위로 변환합니다.
3. 그 Z 구간과 허용 grasp 방향으로 6D map을 slice합니다.
4. 데크·바닥·자기충돌을 필터링합니다.
5. XY 평면으로 투영해 내비게이션팀에 전달할 2D footprint 또는 보수적인 `R_MIN/R_MAX`를 구합니다.

샘플 수를 늘릴 때는 먼저 `NUM_SAMPLES = 1_000_000`으로 확인하고, 최종 실행에서 방향 해상도를 `ANGULAR_RES_DEG = 22.5`로 변경합니다.